# August 5, 2024 Flash Crash Analysis
Analysis of price and gas volatility on 'Black Monday'.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Load data
df = pd.read_csv('../data/aug_5_crash_data.csv')
df['Hour_UTC'] = df['Hour_UTC'].astype(str)
df.head()

## 1. Price vs. Gas Performance

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))

ax1.set_xlabel('Hour (UTC)')
ax1.set_ylabel('ETH Price (USD)', color='tab:blue')
ax1.plot(df['Hour_UTC'], df['Price_USD'], color='tab:blue', marker='o', label='ETH Price')
ax1.tick_params(axis='y', labelcolor='tab:blue')
ax1.set_title('August 5th: Price Crash vs Gas Spike')

ax2 = ax1.twinx()
ax2.set_ylabel('Gas (Gwei)', color='tab:red')
ax2.plot(df['Hour_UTC'], df['Gas_Gwei'], color='tab:red', marker='x', linestyle='--', label='Gas Price')
ax2.tick_params(axis='y', labelcolor='tab:red')

plt.xticks(rotation=45)
fig.tight_layout()
plt.show()

## 2. Parameter Sweep: Success Rate Heatmap
Visualize which LTV/Buffer combinations survived vs liquidated.

In [ ]:
sweep_df = pd.read_csv('../reports/sweep_results.csv')
heatmap_data = sweep_df[sweep_df['Strategy'] == 'static'].pivot(index='Initial_LTV', columns='Buffer', values='Status')

# Convert status to numeric for plotting (1 = sl_triggered, 0 = liquidated)
numeric_status = heatmap_data.replace({'sl_triggered': 1, 'liquidated': 0, 'active': 1})

plt.figure(figsize=(10, 6))
sns.heatmap(numeric_status, annot=heatmap_data, fmt='', cmap='RdYlGn', cbar=False)
plt.title('Aave V3: Success (SL Triggered) vs Failure (Liquidated)\nAug 5, 2024 Crash')
plt.show()

## 3. Equity Preserved: Static vs Dynamic
How much capital was saved using a gas-aware trigger?

In [ ]:
compare_df = sweep_df.groupby(['Initial_LTV', 'Strategy'])['Final_Value_USD'].mean().unstack()
compare_df.plot(kind='bar', figsize=(12, 6))
plt.title('Equity Preservation by Strategy (Aave V3)')
plt.ylabel('Remaining Equity (USD)')
plt.xlabel('Initial LTV')
plt.legend(title='Strategy')
plt.show()